# Lab 04 Lab 04: Data Unions and Joins Pipeline

Santiago Elí Jiménez Aguilar 

In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Lab 04 - Data Unions and Joins"

su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 00:56:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkUtils: Lab 04 - Data Unions and Joins>

In [2]:
#Schemas

agencies_schema = SparkUtils.generate_schema([
    ("agency_id",   "int"),
    ("agency_info", "string"),
])

brands_schema = SparkUtils.generate_schema([
    ("brand_id",   "int"),
    ("brand_info", "string"),
])

cars_schema = SparkUtils.generate_schema([
    ("car_id",   "int"),
    ("car_info", "string"),
])

customers_schema = SparkUtils.generate_schema([
    ("customer_id",   "int"),
    ("customer_info", "string"),
])

rentals_schema = SparkUtils.generate_schema([
    ("rental_id",   "int"),
    ("rental_info", "string"),
])

In [3]:
#Read csv

agencies_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(agencies_schema) \
                .csv("/opt/spark/work-dir/data/car_service/agencies")

brands_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(brands_schema) \
                .csv("/opt/spark/work-dir/data/car_service/brands")

cars_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(cars_schema) \
                .csv("/opt/spark/work-dir/data/car_service/cars")

customers_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(customers_schema) \
                .csv("/opt/spark/work-dir/data/car_service/customers")

rentals_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(rentals_schema) \
                .csv("/opt/spark/work-dir/data/car_service/rentals")

In [4]:
from pyspark.sql.functions import get_json_object, col

agencies_df_ = agencies_df.withColumn("agency_name", get_json_object(agencies_df.agency_info, "$.agency_name"))

cars_df_ = cars_df.withColumn("car_name", get_json_object(cars_df.car_info, "$.car_name"))

customers_df_ = customers_df.withColumn("customer_name", get_json_object(customers_df.customer_info, "$.customer_name"))

rentals_final_df = rentals_df \
    .withColumn("car_id", get_json_object(rentals_df.rental_info, "$.car_id").cast("int")) \
    .withColumn("customer_id", get_json_object(rentals_df.rental_info, "$.customer_id").cast("int")) \
    .withColumn("agency_id", get_json_object(rentals_df.rental_info, "$.agency_id").cast("int"))

In [5]:
rentals_final_df = rentals_final_df \
    .join(cars_df_, on="car_id", how="inner") \
    .join(agencies_df_, on="agency_id", how="inner") \
    .join(customers_df_, on="customer_id", how="inner") \
    .select("rental_id", "car_name", "agency_name", "customer_name")

In [6]:
rentals_final_df.show(truncate=False)

+---------+-----------------------------------+-------------+---------------+
|rental_id|car_name                           |agency_name  |customer_name  |
+---------+-----------------------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9            |NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8               |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5              |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4                 |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1                 |SF Cars      |Kristin Potts  |
|11896    |Jones, Jefferson and Rivera Model 7|LA Car Rental|Jeremy Parks   |
|11897    |Lopez and Sons Model 9             |Zapopan Auto |Terry Wells    |
|11898    |Salazar Ltd Model 8                |SF Cars      |Marc Williams  |
|11899    |Villanueva PLC Model 7             |LA Car Rental|Danny Williams |
|11900    |Faulkner-Howard Model 5            |SF Cars      |Eri

In [7]:
## SAVE INTO STORAGE

rentals_final_df.write \
    .partitionBy("agency_name") \
    .mode("overwrite") \
    .parquet("hdfs://namenode:9000/output/agencies_car_service")

In [8]:
su.spark_context().stop()